# Personalization evaluation file generator

This notebook generates CSV templates for evaluating personalized datasets (3/20/25)

## Variable level ratings

The CSV was brought into Excel, where

- Rating validation was set to `Strongly Disagree, Disagree, Somewhat Disagree, Somewhat Agree, Agree, Strongly Agree`
- Color was added to topic, variable name, and levels columns
- Light gray was added to the ratings columns
- The file id columns was removed

In [37]:
match_question = "The variable name matches the topic"
range_question = "The range of the variable is plausible based on your knowledge"
level_question = "The levels of the variable are plausible based on your knowledge"

notheme_map = {
    "Stats0.csv":"iris",
    "KNNReg0.csv":"boston",
    "RegTrees0.csv":"baseball",
    "Forest0.csv":"cancer",
    "DecTree0.csv":"candy"
}

import os
import pandas as pd
import base64

# collect rows for output
rows = []

# we are dropping News from evaluation for efficiency
topics = ['NoTheme', 'Travel','Sports', 'Games', 'Health']

# get tuples of topic, filehash, var name,   match question, rating, min/median/max OR levels, range question, rating
for topic in topics:
    for filename in os.listdir("datasets/" + topic):
        if filename.endswith(".csv"):
            path = "datasets/" + topic + "/" + filename
            df = pd.read_csv(path)
            col_names = df.columns
            
            # obfuscate file name to remove bias
            file_hash = base64.b64encode(filename.encode("ascii")).decode("ascii") #hash(filename)
            
            # fix notheme names
            for var_name in col_names:
                if filename in notheme_map:
                        adjusted_topic = notheme_map[filename]
                else:
                    adjusted_topic = topic
                
                # numeric question types
                if pd.api.types.is_numeric_dtype(df[var_name]):
                    a_min = df[var_name].min()
                    a_median = df[var_name].median()
                    a_max = df[var_name].max()
                    if df[var_name].astype(str).str.isdigit().all(): #pd.api.types.is_float_dtype:
                        range_string = f"{a_min}--{a_max}"
                        # range_string = f"{a_min:.2f}--{a_max:.2f}"
                    else:
                        # range_string = f"{a_min}--{a_max}"
                        range_string = f"{a_min:.2f}--{a_max:.2f}"
                    row = (adjusted_topic, filename, file_hash, var_name, match_question, "", range_string, range_question, "" )
                else:
                    a_levels = df[var_name].unique().tolist()
                    row = (adjusted_topic, filename,file_hash, var_name, match_question, "", f"{a_levels}", level_question, "" )
                rows.append(row)

output = pd.DataFrame(rows, columns =['topic','filename','filehash','var name','match question','rating','min/max OR levels','range/level question', 'rating'])
output.to_csv("eval_sheet_variable.csv",index=False)

## Notebook level ratings

In [3]:
import os
import pandas as pd
import base64

question = """Count the number of "problems" in the notebook.
Examples:
- spaghetti plots
- bar plot with a variable that appears to be missing/zero, but is actually just very small
- class that has only 1 level instead of at least 2
- text referring to NA that aren't there
- text referring to linear/nonlinear patterns in plots that aren't there"""

rows = []
# html approach
# files = os.listdir("html/variations")
# for f in files:
#     filename = f.replace('.html','')

# notebook approach
notebooks = ['Descriptive-statistics.ipynb','KNN-regression.ipynb','Random-forests.ipynb','Regression-trees.ipynb']
topics = ['NoTheme', 'Travel','Sports', 'Games', 'Health']
for n in notebooks:
    for t in topics:
        # no theme has method 0
        if t.startswith("No"):
            row = (n, t, "0", question, "" )
            rows.append(row)
        else:
            row = (n, t, "1", question, "" )
            rows.append(row)
            row = (n, t, "2", question, "" )
            rows.append(row)
            
output = pd.DataFrame(rows, columns =['notebook','topic','method','question',"problem count"])
output.to_csv("eval_sheet_html.csv",index=False)